# 02 - Pré-processamento de Dados
## Tech Challenge Fase 1 - Saúde e Segurança da Mulher
**Responsável:** Natalia Cabrera

---

### Objetivo
Aplicar o pipeline de pré-processamento nos dados, preparando-os para a modelagem de Machine Learning.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append("../..")

from src.preprocessing import (
    check_data_quality, handle_missing_values, encode_categorical,
    scale_features, split_data, full_preprocessing_pipeline
)
from src.pipeline import load_wisconsin_dataset

import warnings
warnings.filterwarnings("ignore")
print("Imports carregados!")

## 1. Carregamento e Verificação de Qualidade

In [ ]:
# Carregar dados do novo dataset (Complementar Tumoral)
# Corrige automaticamente valores inteiros (÷100)
df = load_wisconsin_dataset()

# Verificar qualidade
quality_report = check_data_quality(df)

# Verificar se valores estão corretos
print(f"
Verificação dos valores (devem ser floats):")
print(f"  radius_mean: {df["radius_mean"].min():.2f} - {df["radius_mean"].max():.2f}")
print(f"  area_mean: {df["area_mean"].min():.2f} - {df["area_mean"].max():.2f}")

## 2. Pipeline Completo de Pré-processamento

In [ ]:
# Executar pipeline completo
pipeline_result = full_preprocessing_pipeline(
    df,
    target_col="target",
    drop_cols=["diagnosis"],
    test_size=0.2
)

# Extrair resultados
X_train = pipeline_result["X_train"]
X_test = pipeline_result["X_test"]
y_train = pipeline_result["y_train"]
y_test = pipeline_result["y_test"]
feature_names = pipeline_result["feature_names"]
scaler = pipeline_result["scaler"]

print(f"
Features: {len(feature_names)}")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

In [ ]:
# Verificar distribuições após padronização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Antes (primeiras 5 features originais)
df[feature_names[:5]].boxplot(ax=axes[0])
axes[0].set_title('Antes da Padronização (5 primeiras features)')
axes[0].tick_params(axis='x', rotation=45)

# Depois
X_train[feature_names[:5]].boxplot(ax=axes[1])
axes[1].set_title('Após Padronização (StandardScaler)')
axes[1].tick_params(axis='x', rotation=45)

import os
# Garantir que o diretório de figuras exista (caminho relativo ao notebook)
os.makedirs('../../reports/figures', exist_ok=True)
plt.tight_layout()
plt.savefig('../../reports/figures/padronizacao_comparacao.png', dpi=150)
plt.show()

In [ ]:
# Salvar dados processados
import os
# Garantir que o diretório de dados processados exista (caminho relativo ao notebook)
os.makedirs('../../data/processed', exist_ok=True)
X_train.to_csv('../../data/processed/X_train.csv', index=False)
X_test.to_csv('../../data/processed/X_test.csv', index=False)
y_train.to_csv('../../data/processed/y_train.csv', index=False)
y_test.to_csv('../../data/processed/y_test.csv', index=False)

print("Dados processados salvos em data/processed/")

## 3. Resumo do Pré-processamento

**Resumo aplicado pelo pipeline `full_preprocessing_pipeline`**

| Etapa | Decisão tomada | Justificativa |
|---|---|---|
| Valores ausentes | Nenhum (0 valores faltantes) | Dataset originalmente limpo; não foi necessária imputação |
| Codificação | `diagnosis` removido; target usado: `target` (numérico) | `target` já presente no dataset; evita duplicidade textual |
| Padronização | `StandardScaler` aplicado às features numéricas | Recomendado para modelos sensíveis à escala (SVM, LR, NN); preserva relação entre variáveis |
| Split | 80/20 estratificado (train/test) | Mantém proporção das classes entre conjuntos para validação robusta |
| Colunas removidas | `diagnosis` (removida conforme parâmetro `drop_cols`) | Remove redundância com `target`; nenhuma outra coluna foi removida automaticamente |

**Observações**: Os conjuntos resultantes estão em `X_train`, `X_test`, `y_train`, `y_test`. Arquivos gerados: figuras em `reports/figures` e CSVs em `data/processed`.

*(Atualize esta seção se executar remoções adicionais por correlação/VIF/PCA)*